# 06 Music albums — v37 direct-record queue + progress fix

This patch fixes the v35 issue where the notebook appeared to create only a candidate JSON and then iterated a huge `music specs` progress bar before producing any benchmark records.

Main changes:
- candidate facets are now **in-memory only**; no candidate queue JSON is written to `domain_outputs`;
- generation is **per complexity level** with a progress bar over accepted target records, not over thousands of candidates;
- candidate ordering is rewritten so fast/high-yield release records are tried first;
- records are still written incrementally to `music_albums.jsonl` immediately after acceptance;
- old audit reject signatures are ignored when the generator version changes;
- gold answers are still collected directly from WDQS with count diagnostics, not from any local cache.


In [1]:
# -------------------------
# Imports and configuration
# -------------------------
from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple, Set
import datetime as _dt
import hashlib
import json
import os
import random
import re
import time

import requests
from tqdm.auto import tqdm

# Try to reuse project OUT_DIR if available, otherwise use the standard local path.
try:
    OUT_DIR
except NameError:
    OUT_DIR = "out_wikidata_benchmark"

MUSIC_DOMAIN_NAME = "music_albums"
MUSIC_GENERATOR_VERSION = "music_albums_v37_fast_single_query_golds"
DOMAIN_OUT_DIR = Path(OUT_DIR) / "domain_outputs"
DOMAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)

MUSIC_OUTPUT_PATH = DOMAIN_OUT_DIR / "music_albums.jsonl"
MUSIC_AUDIT_PATH = DOMAIN_OUT_DIR / "music_albums_generation_audit.json"
MUSIC_CHECKPOINT_PATH = DOMAIN_OUT_DIR / "music_albums_generation_checkpoint.json"
MUSIC_CANDIDATE_CACHE_PATH = None  # v36: no candidate queue JSON; candidates are in-memory only

# Target: high-quality 100–120 records.
MUSIC_TARGET_PER_LEVEL = {"L1": 15, "L2": 20, "L3": 25, "L4": 25, "L5": 20}
MUSIC_TARGET_TOTAL = sum(MUSIC_TARGET_PER_LEVEL.values())
REQUESTED_BY_LEVEL = {"L1": 5, "L2": 5, "L3": 5, "L4": 4, "L5": 3}
MIN_GOLD_BY_LEVEL = {"L1": 7, "L2": 7, "L3": 6, "L4": 5, "L5": 5}
MAX_GOLD_BY_LEVEL = {"L1": 70, "L2": 60, "L3": 50, "L4": 45, "L5": 40}

# Desired answer-kind balance. Albums intentionally dominate singles.
MUSIC_KIND_TARGET_SOFT = {"studio album": 45, "EP": 20, "single": 15, "performer": 25}
MUSIC_KIND_MAX = {"studio album": 58, "EP": 30, "single": 22, "performer": 35}
MUSIC_LEVEL_KIND_MAX = {
    "L1": {"studio album": 10, "EP": 5, "single": 4, "performer": 0},
    "L2": {"studio album": 12, "EP": 6, "single": 5, "performer": 0},
    "L3": {"studio album": 12, "EP": 7, "single": 6, "performer": 10},
    "L4": {"studio album": 10, "EP": 8, "single": 5, "performer": 12},
    "L5": {"studio album": 8, "EP": 6, "single": 5, "performer": 12},
}

MUSIC_SEED = 20260528
_rng = random.Random(MUSIC_SEED)

# WDQS settings. A failed/slow candidate is skipped; the generator should not hang on one SPARQL.
WDQS_ENDPOINT = "https://query.wikidata.org/sparql"
WDQS_TIMEOUT_SECONDS = 16
WDQS_MAX_RETRIES = 0
WDQS_SLEEP_SECONDS = 0.18
WDQS_USER_AGENT = "music-multihop-benchmark-v37-fast-single-query/0.1 (https://chat.openai.com)"

# Completeness policy: if Wikidata has matching QIDs without English labels, skip the task.
# This keeps gold_answer_labels_en complete and avoids hidden answers.
MUSIC_STRICT_EN_LABEL_COMPLETENESS = True
MUSIC_FORCE_REBUILD_CANDIDATE_QUEUE = False
MUSIC_ALLOW_PARTIAL_OUTPUT = True
MAX_ATTEMPTS_PER_LEVEL = {"L1": 260, "L2": 420, "L3": 520, "L4": 620, "L5": 620}

# Diversity / dedup.
TEMPLATE_CAP_PER_LEVEL = {"L1": 5, "L2": 5, "L3": 6, "L4": 6, "L5": 5}
TEMPLATE_FAMILY_CAP_PER_LEVEL = {"L1": 7, "L2": 8, "L3": 9, "L4": 9, "L5": 8}
VISIBLE_VALUE_CAP_TOTAL = 8
GOLD_JACCARD_REJECT = 0.78
GOLD_CONTAINMENT_REJECT = 0.92

print("output:", MUSIC_OUTPUT_PATH.resolve())
print("audit:", MUSIC_AUDIT_PATH.resolve())
print("candidate queue:", "in-memory only; benchmark records go to", MUSIC_OUTPUT_PATH.resolve())
print("target:", MUSIC_TARGET_PER_LEVEL, "total", MUSIC_TARGET_TOTAL)


output: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/music_albums.jsonl
audit: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/music_albums_generation_audit.json
candidate queue: in-memory only; benchmark records go to /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/music_albums.jsonl
target: {'L1': 15, 'L2': 20, 'L3': 25, 'L4': 25, 'L5': 20} total 105


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------------------
# File and WDQS helpers
# -------------------------
def _json_dump(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def _json_load(path: Path, default: Any = None) -> Any:
    if not path.exists():
        return default
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def _read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                raise ValueError(f"Bad JSONL line {line_no} in {path}: {e}")
    return rows


def _append_jsonl(path: Path, row: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(tmp, path)


def norm_text(s: Any) -> str:
    return re.sub(r"\s+", " ", str(s or "").strip()).lower()


def clean_label(s: Any) -> str:
    return re.sub(r"\s+", " ", str(s or "").strip())


def constraints_signature(c: Dict[str, Any]) -> str:
    return json.dumps(c, ensure_ascii=False, sort_keys=True)


def safe_year_expr(var: str) -> str:
    return f"BIND(YEAR({var}) AS ?_year_{var.strip('?')}) ."


def q(qid: str) -> str:
    assert re.match(r"^Q\d+$", qid), qid
    return f"wd:{qid}"


def escape_sparql_string(s: str) -> str:
    return str(s).replace('\\', '\\\\').replace('"', '\\"')


_wdqs_cache: Dict[str, Dict[str, Any]] = {}


def wdqs(query: str, *, timeout: int = WDQS_TIMEOUT_SECONDS, use_cache: bool = True) -> Dict[str, Any]:
    key = hashlib.sha1(query.encode("utf-8")).hexdigest()
    if use_cache and key in _wdqs_cache:
        return _wdqs_cache[key]
    headers = {"Accept": "application/sparql-results+json", "User-Agent": WDQS_USER_AGENT}
    last_error = None
    for attempt in range(WDQS_MAX_RETRIES + 1):
        try:
            resp = requests.get(WDQS_ENDPOINT, params={"query": query, "format": "json"}, headers=headers, timeout=timeout)
            if resp.status_code == 429:
                time.sleep(2.0 + attempt)
                continue
            resp.raise_for_status()
            data = resp.json()
            if use_cache:
                _wdqs_cache[key] = data
            time.sleep(WDQS_SLEEP_SECONDS)
            return data
        except Exception as e:
            last_error = e
            if attempt < WDQS_MAX_RETRIES:
                time.sleep(1.0 + attempt)
    raise RuntimeError(f"WDQS failed after retries: {last_error}")


def wd_value(binding: Dict[str, Any], key: str, default: str = "") -> str:
    v = binding.get(key, {})
    return v.get("value", default) if isinstance(v, dict) else default


def wd_qid_from_uri(uri: str) -> str:
    m = re.search(r"/(Q\d+)$", str(uri))
    return m.group(1) if m else str(uri)


def parse_count(data: Dict[str, Any], var: str = "count") -> int:
    rows = data.get("results", {}).get("bindings", [])
    if not rows:
        return 0
    return int(float(wd_value(rows[0], var, "0")))


In [3]:
# -------------------------
# QIDs, labels and reusable SPARQL snippets
# -------------------------
RELEASE_KINDS = {
    # Internal key kept as "studio album" for compatibility with the existing quota code.
    # In Wikidata, many music albums are modeled only as generic albums (Q482994), not as
    # direct instances/subclasses of studio album (Q208569). Therefore v34 asks for
    # "music albums" and uses a robust album pattern instead of the too-narrow Q208569-only pattern.
    "studio album": {"qid": "Q482994", "ru": "музыкальных альбомов", "en_plural": "music albums", "en_single": "music album"},
    "EP": {"qid": "Q169930", "ru": "мини-альбомов (EP)", "en_plural": "EPs", "en_single": "EP"},
    "single": {"qid": "Q134556", "ru": "синглов", "en_plural": "singles", "en_single": "single"},
}

# Fallback curated facets. Discovery queries will add more facets dynamically.
CURATED_GENRES = [
    {"qid": "Q11399", "label_en": "rock music", "label_ru": "рок"},
    {"qid": "Q37073", "label_en": "pop music", "label_ru": "поп-музыка"},
    {"qid": "Q11401", "label_en": "hip hop music", "label_ru": "хип-хоп"},
    {"qid": "Q9778", "label_en": "electronic music", "label_ru": "электронная музыка"},
    {"qid": "Q8341", "label_en": "jazz", "label_ru": "джаз"},
    {"qid": "Q38848", "label_en": "heavy metal", "label_ru": "хеви-метал"},
    {"qid": "Q9759", "label_en": "blues", "label_ru": "блюз"},
    {"qid": "Q9794", "label_en": "reggae", "label_ru": "регги"},
    {"qid": "Q9730", "label_en": "classical music", "label_ru": "классическая музыка"},
    {"qid": "Q131272", "label_en": "soul music", "label_ru": "соул"},
    {"qid": "Q10922", "label_en": "punk rock", "label_ru": "панк-рок"},
    {"qid": "Q45981", "label_en": "rhythm and blues", "label_ru": "ритм-н-блюз"},
]
CURATED_LANGUAGES = [
    {"qid": "Q1860", "label_en": "English", "label_ru": "английский язык"},
    {"qid": "Q1321", "label_en": "Spanish", "label_ru": "испанский язык"},
    {"qid": "Q150", "label_en": "French", "label_ru": "французский язык"},
    {"qid": "Q188", "label_en": "German", "label_ru": "немецкий язык"},
    {"qid": "Q652", "label_en": "Italian", "label_ru": "итальянский язык"},
    {"qid": "Q5287", "label_en": "Japanese", "label_ru": "японский язык"},
    {"qid": "Q9176", "label_en": "Korean", "label_ru": "корейский язык"},
    {"qid": "Q5146", "label_en": "Portuguese", "label_ru": "португальский язык"},
    {"qid": "Q7737", "label_en": "Russian", "label_ru": "русский язык"},
]
CURATED_COUNTRIES = [
    {"qid": "Q30", "label_en": "United States", "label_ru": "США"},
    {"qid": "Q145", "label_en": "United Kingdom", "label_ru": "Великобритания"},
    {"qid": "Q16", "label_en": "Canada", "label_ru": "Канада"},
    {"qid": "Q183", "label_en": "Germany", "label_ru": "Германия"},
    {"qid": "Q142", "label_en": "France", "label_ru": "Франция"},
    {"qid": "Q17", "label_en": "Japan", "label_ru": "Япония"},
    {"qid": "Q884", "label_en": "South Korea", "label_ru": "Южная Корея"},
    {"qid": "Q155", "label_en": "Brazil", "label_ru": "Бразилия"},
    {"qid": "Q29", "label_en": "Spain", "label_ru": "Испания"},
    {"qid": "Q34", "label_en": "Sweden", "label_ru": "Швеция"},
    {"qid": "Q408", "label_en": "Australia", "label_ru": "Австралия"},
]

CURATED_RECORD_LABELS = [
    {"qid": "Q183387", "label_en": "Columbia Records", "label_ru": "Columbia Records"},
    {"qid": "Q202440", "label_en": "Atlantic Records", "label_ru": "Atlantic Records"},
    {"qid": "Q208909", "label_en": "Parlophone", "label_ru": "Parlophone"},
    {"qid": "Q193023", "label_en": "Capitol Records", "label_ru": "Capitol Records"},
    {"qid": "Q43327", "label_en": "Motown", "label_ru": "Motown"},
    {"qid": "Q885833", "label_en": "Blue Note", "label_ru": "Blue Note"},
    {"qid": "Q64485314", "label_en": "Warner Records", "label_ru": "Warner Records"},
    {"qid": "Q3629023", "label_en": "EMI Records", "label_ru": "EMI Records"},
    {"qid": "Q1123947", "label_en": "XL Recordings", "label_ru": "XL Recordings"},
    {"qid": "Q654283", "label_en": "Def Jam Recordings", "label_ru": "Def Jam Recordings"},
]

PERIODS = [
    {"from": 1960, "to": 1980, "label_ru": "1960–1980", "label_en": "1960–1980"},
    {"from": 1981, "to": 1995, "label_ru": "1981–1995", "label_en": "1981–1995"},
    {"from": 1996, "to": 2010, "label_ru": "1996–2010", "label_en": "1996–2010"},
    {"from": 2011, "to": 2025, "label_ru": "2011–2025", "label_en": "2011–2025"},
]


def release_kind_pattern(var: str, kind: str) -> str:
    # v34 fix: the old Q208569-only studio-album pattern produced zero candidates on WDQS.
    # Use the broader music album class for album tasks, while excluding EP/single classes.
    if kind == "studio album":
        return f"""{var} wdt:P31/wdt:P279* wd:Q482994 .
      FILTER NOT EXISTS {{ {var} wdt:P31/wdt:P279* wd:Q169930 . }}
      FILTER NOT EXISTS {{ {var} wdt:P31/wdt:P279* wd:Q134556 . }}"""
    return f"{var} wdt:P31/wdt:P279* {q(RELEASE_KINDS[kind]['qid'])} ."


def date_filter(var: str, y1: int, y2: int, date_var: str) -> str:
    return f"{var} wdt:P577 {date_var} .\n      BIND(YEAR({date_var}) AS ?year_{date_var.strip('?')}) .\n      FILTER(?year_{date_var.strip('?')} >= {int(y1)} && ?year_{date_var.strip('?')} <= {int(y2)}) ."


def label_clause(var: str = "?item", en_var: str = "?itemLabelEn", ru_var: str = "?itemLabelRu") -> str:
    return f"{var} rdfs:label {en_var} FILTER(LANG({en_var}) = \"en\") .\n      OPTIONAL {{ {var} rdfs:label {ru_var} FILTER(LANG({ru_var}) = \"ru\") . }}"


def facet_label(facet: Dict[str, Any], lang: str = "en") -> str:
    return clean_label(facet.get(f"label_{lang}") or facet.get("label_en") or facet.get("qid"))


def period_constraint(p: Dict[str, int]) -> Dict[str, int]:
    return {"publication_year_min": int(p["from"]), "publication_year_max": int(p["to"])}


In [4]:
# -------------------------
# Candidate discovery from WDQS
# -------------------------
# Candidate discovery may be cached, but gold answers are never taken from this cache.

def discover_facets_for_kind_period(kind: str, facet: str, period: Dict[str, int], limit: int = 40) -> List[Dict[str, Any]]:
    """Discover useful genre/label/language/country facets by grouped WDQS counts."""
    kind_pat = release_kind_pattern("?item", kind)
    date_pat = date_filter("?item", int(period["from"]), int(period["to"]), "?pub_date")
    if facet == "genre":
        facet_pat = "?item wdt:P136 ?facet ."
    elif facet == "record_label":
        facet_pat = "?item wdt:P264 ?facet ."
    elif facet == "language":
        facet_pat = "?item wdt:P407 ?facet ."
    elif facet == "performer_country":
        facet_pat = "?item wdt:P175 ?performer .\n      { ?performer wdt:P27 ?facet . } UNION { ?performer wdt:P495 ?facet . } UNION { ?performer wdt:P17 ?facet . }"
    elif facet == "performer_genre":
        facet_pat = "?item wdt:P175 ?performer .\n      ?performer wdt:P136 ?facet ."
    else:
        raise ValueError(facet)
    query = f"""
SELECT ?facet ?facetLabelEn ?facetLabelRu (COUNT(DISTINCT ?item) AS ?count) WHERE {{
      {kind_pat}
      {date_pat}
      {facet_pat}
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      ?facet rdfs:label ?facetLabelEn FILTER(LANG(?facetLabelEn) = "en") .
      OPTIONAL {{ ?facet rdfs:label ?facetLabelRu FILTER(LANG(?facetLabelRu) = "ru") . }}
}}
GROUP BY ?facet ?facetLabelEn ?facetLabelRu
HAVING(COUNT(DISTINCT ?item) >= 6 && COUNT(DISTINCT ?item) <= 80)
ORDER BY DESC(?count)
LIMIT {int(limit)}
"""
    try:
        data = wdqs(query, timeout=min(WDQS_TIMEOUT_SECONDS, 18))
    except Exception as e:
        print(f"discover {kind}/{facet}/{period['from']}-{period['to']} failed:", e)
        return []
    out = []
    for b in data.get("results", {}).get("bindings", []):
        out.append({
            "qid": wd_qid_from_uri(wd_value(b, "facet")),
            "label_en": clean_label(wd_value(b, "facetLabelEn")),
            "label_ru": clean_label(wd_value(b, "facetLabelRu") or wd_value(b, "facetLabelEn")),
            "count_hint": int(float(wd_value(b, "count", "0"))),
            "facet": facet,
        })
    return out


def discover_seed_releases(limit: int = 140) -> List[Dict[str, Any]]:
    """Find releases that have a label and genre where same-label bridge has enough answers."""
    query = f"""
SELECT ?seed ?seedLabelEn ?seedLabelRu ?label ?labelLabelEn ?labelLabelRu ?genre ?genreLabelEn ?genreLabelRu (COUNT(DISTINCT ?other) AS ?same_label_count) WHERE {{
      ?seed wdt:P31/wdt:P279* wd:Q208569 .
      ?seed wdt:P264 ?label .
      ?seed wdt:P136 ?genre .
      ?seed rdfs:label ?seedLabelEn FILTER(LANG(?seedLabelEn) = "en") .
      OPTIONAL {{ ?seed rdfs:label ?seedLabelRu FILTER(LANG(?seedLabelRu) = "ru") . }}
      ?label rdfs:label ?labelLabelEn FILTER(LANG(?labelLabelEn) = "en") .
      OPTIONAL {{ ?label rdfs:label ?labelLabelRu FILTER(LANG(?labelLabelRu) = "ru") . }}
      ?genre rdfs:label ?genreLabelEn FILTER(LANG(?genreLabelEn) = "en") .
      OPTIONAL {{ ?genre rdfs:label ?genreLabelRu FILTER(LANG(?genreLabelRu) = "ru") . }}
      ?other wdt:P31/wdt:P279* wd:Q208569 .
      ?other wdt:P264 ?label .
      ?other rdfs:label ?otherLabelEn FILTER(LANG(?otherLabelEn) = "en") .
      FILTER(?other != ?seed)
}}
GROUP BY ?seed ?seedLabelEn ?seedLabelRu ?label ?labelLabelEn ?labelLabelRu ?genre ?genreLabelEn ?genreLabelRu
HAVING(COUNT(DISTINCT ?other) >= 6 && COUNT(DISTINCT ?other) <= 60)
ORDER BY DESC(?same_label_count)
LIMIT {int(limit)}
"""
    try:
        data = wdqs(query, timeout=WDQS_TIMEOUT_SECONDS + 15)
    except Exception as e:
        print("discover seed releases failed:", e)
        return []
    out = []
    for b in data.get("results", {}).get("bindings", []):
        out.append({
            "qid": wd_qid_from_uri(wd_value(b, "seed")),
            "label_en": clean_label(wd_value(b, "seedLabelEn")),
            "label_ru": clean_label(wd_value(b, "seedLabelRu") or wd_value(b, "seedLabelEn")),
            "record_label_qid": wd_qid_from_uri(wd_value(b, "label")),
            "record_label_en": clean_label(wd_value(b, "labelLabelEn")),
            "record_label_ru": clean_label(wd_value(b, "labelLabelRu") or wd_value(b, "labelLabelEn")),
            "genre_qid": wd_qid_from_uri(wd_value(b, "genre")),
            "genre_en": clean_label(wd_value(b, "genreLabelEn")),
            "genre_ru": clean_label(wd_value(b, "genreLabelRu") or wd_value(b, "genreLabelEn")),
            "same_label_count_hint": int(float(wd_value(b, "same_label_count", "0"))),
        })
    return out


def build_candidate_facet_cache(force_rebuild: bool = MUSIC_FORCE_REBUILD_CANDIDATE_QUEUE) -> Dict[str, Any]:
    """Build a deterministic local candidate facet cache without WDQS discovery.

    v33/v34 used heavy GROUP BY discovery queries over all music releases. Those queries are
    exactly what caused the notebook to time out before generation. In v35, discovery is
    deliberately disabled: the queue is built from curated stable facets, while the final golds
    are still collected and counted directly in WDQS for every accepted record.
    """
    if MUSIC_CANDIDATE_CACHE_PATH.exists() and not force_rebuild:
        cache = _json_load(MUSIC_CANDIDATE_CACHE_PATH, {})
        if cache.get("version") == MUSIC_GENERATOR_VERSION:
            print("loaded no-discovery candidate cache:", MUSIC_CANDIDATE_CACHE_PATH)
            return cache
    cache: Dict[str, Any] = {
        "version": MUSIC_GENERATOR_VERSION,
        "created_at": _dt.datetime.utcnow().isoformat()+"Z",
        "mode": "no_wdqs_facet_discovery_curated_facets_only",
        "facets": {},
        "seeds": [],
    }
    for kind in RELEASE_KINDS:
        for period in PERIODS:
            key_prefix = f"{kind}|{period['from']}|{period['to']}"
            cache["facets"][f"{key_prefix}|genre"] = [{**x, "facet": "genre"} for x in CURATED_GENRES]
            cache["facets"][f"{key_prefix}|record_label"] = [{**x, "facet": "record_label"} for x in CURATED_RECORD_LABELS]
            cache["facets"][f"{key_prefix}|language"] = [{**x, "facet": "language"} for x in CURATED_LANGUAGES]
            cache["facets"][f"{key_prefix}|performer_country"] = [{**x, "facet": "performer_country"} for x in CURATED_COUNTRIES]
            cache["facets"][f"{key_prefix}|performer_genre"] = [{**x, "facet": "performer_genre"} for x in CURATED_GENRES]
    print("candidate facet cache built locally: no WDQS discovery; facet keys:", len(cache["facets"]))
    _json_dump(MUSIC_CANDIDATE_CACHE_PATH, cache)
    return cache

def get_facets(cache: Dict[str, Any], kind: str, period: Dict[str, int], facet: str, fallback: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:
    key = f"{kind}|{period['from']}|{period['to']}|{facet}"
    rows = list(cache.get("facets", {}).get(key, []) or [])
    if fallback:
        seen = {r.get("qid") for r in rows}
        for r in fallback:
            if r.get("qid") not in seen:
                rows.append({**r, "facet": facet})
                seen.add(r.get("qid"))
    return rows


# --- v36 override: no candidate cache file, no WDQS discovery, in-memory facets only ---
def build_candidate_facet_cache(force_rebuild: bool = False) -> Dict[str, Any]:
    """Build deterministic candidate facets in memory only.

    v36 deliberately does not write music_albums_candidate_queue_*.json. The only growing
    benchmark artifact during generation is MUSIC_OUTPUT_PATH (music_albums.jsonl), plus audit/checkpoint.
    """
    cache: Dict[str, Any] = {
        "version": MUSIC_GENERATOR_VERSION,
        "created_at": _dt.datetime.utcnow().isoformat()+"Z",
        "mode": "v36_in_memory_curated_facets_only_no_candidate_json",
        "facets": {},
        "seeds": [],
    }
    for kind in RELEASE_KINDS:
        for period in PERIODS:
            key_prefix = f"{kind}|{period['from']}|{period['to']}"
            cache["facets"][f"{key_prefix}|genre"] = [{**x, "facet": "genre"} for x in CURATED_GENRES]
            cache["facets"][f"{key_prefix}|record_label"] = [{**x, "facet": "record_label"} for x in CURATED_RECORD_LABELS]
            cache["facets"][f"{key_prefix}|language"] = [{**x, "facet": "language"} for x in CURATED_LANGUAGES]
            cache["facets"][f"{key_prefix}|performer_country"] = [{**x, "facet": "performer_country"} for x in CURATED_COUNTRIES]
            cache["facets"][f"{key_prefix}|performer_genre"] = [{**x, "facet": "performer_genre"} for x in CURATED_GENRES]
    print("candidate facets built in memory: no WDQS discovery, no candidate JSON; facet keys:", len(cache["facets"]))
    return cache


In [5]:
# -------------------------
# SPARQL spec builders
# -------------------------
def spec_hash(spec: Dict[str, Any]) -> int:
    payload = json.dumps({"template_id": spec.get("template_id"), "constraints": spec.get("constraints")}, ensure_ascii=False, sort_keys=True)
    return int(hashlib.md5(payload.encode("utf-8")).hexdigest()[:8], 16)


def release_answer_spec(level: str, kind: str, template_id: str, template_family: str, body_lines: List[str], constraints: Dict[str, Any], query_ru: str, query_en: str, bridge_meta: Dict[str, Any]) -> Dict[str, Any]:
    public_kind = "music album" if kind == "studio album" else kind
    constraints = {"kind": public_kind, **constraints}
    return {
        "level": level,
        "answer_kind": "release",
        "answer_kind_quota_bucket": kind,
        "requested_count": REQUESTED_BY_LEVEL[level],
        "template_id": template_id,
        "template_family": template_family,
        "body": "\n      ".join(body_lines),
        "constraints": constraints,
        "query_text_ru": query_ru,
        "query_text_en": query_en,
        "bridge_meta": bridge_meta,
        "priority": 0,
    }


def performer_answer_spec(level: str, template_id: str, template_family: str, body_lines: List[str], constraints: Dict[str, Any], query_ru: str, query_en: str, bridge_meta: Dict[str, Any]) -> Dict[str, Any]:
    constraints = {"kind": "performer", **constraints}
    return {
        "level": level,
        "answer_kind": "performer",
        "answer_kind_quota_bucket": "performer",
        "requested_count": REQUESTED_BY_LEVEL[level],
        "template_id": template_id,
        "template_family": template_family,
        "body": "\n      ".join(body_lines),
        "constraints": constraints,
        "query_text_ru": query_ru,
        "query_text_en": query_en,
        "bridge_meta": bridge_meta,
        "priority": 0,
    }


def release_gold_queries(spec: Dict[str, Any]) -> Tuple[str, Optional[str], Optional[str]]:
    """Return a single fast SELECT query.

    v36 used three WDQS calls per candidate: COUNT with English label, COUNT without
    label filter, then SELECT. That made even L1 slow. v37 uses one SELECT with
    OPTIONAL EN/RU labels. The same result gives:
      - the complete answer universe without a label filter;
      - the English-label answer universe;
      - the final gold labels.
    If there are matching QIDs without English labels and strict completeness is enabled,
    the candidate is rejected before writing a record.
    """
    body = spec["body"]
    select_query = f"""
SELECT DISTINCT ?item ?itemLabelEn ?itemLabelRu WHERE {{
      {body}
      OPTIONAL {{ ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") . }}
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
}}
LIMIT 701
"""
    return select_query, None, None

def ask_query_for_spec(spec: Dict[str, Any]) -> str:
    body = spec["body"].replace("?item", "?candidate")
    return f"""# WDQS-only validator. Replace {{ITEM}} with a candidate answer QID.
ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?candidate)
      {body}
}}"""


def level_kind_ok(spec: Dict[str, Any], existing: List[Dict[str, Any]]) -> bool:
    level = spec["level"]
    bucket = spec["answer_kind_quota_bucket"]
    if MUSIC_LEVEL_KIND_MAX.get(level, {}).get(bucket, 0) <= 0:
        return False
    if sum(1 for r in existing if r.get("complexity") == level and (r.get("gold_collection_meta") or {}).get("answer_kind_quota_bucket") == bucket) >= MUSIC_LEVEL_KIND_MAX[level][bucket]:
        return False
    if sum(1 for r in existing if (r.get("gold_collection_meta") or {}).get("answer_kind_quota_bucket") == bucket) >= MUSIC_KIND_MAX[bucket]:
        return False
    return True


def visible_values(spec: Dict[str, Any]) -> List[Tuple[str, str]]:
    c = spec.get("constraints", {})
    vals = []
    for k, v in c.items():
        if k in {"kind", "publication_year_min", "publication_year_max"}:
            continue
        if isinstance(v, (str, int)):
            vals.append((k, str(v)))
    return vals


def spec_precheck(spec: Dict[str, Any], existing: List[Dict[str, Any]], rejected_sigs: Set[str]) -> Tuple[bool, str]:
    level = spec["level"]
    if sum(1 for r in existing if r.get("complexity") == level) >= MUSIC_TARGET_PER_LEVEL[level]:
        return False, "level_full"
    sig = constraints_signature(spec["constraints"])
    if sig in rejected_sigs:
        return False, "signature_rejected_before"
    if any(constraints_signature(r.get("constraints", {})) == sig for r in existing):
        return False, "duplicate_constraints"
    if sum(1 for r in existing if r.get("complexity") == level and r.get("template_id") == spec.get("template_id")) >= TEMPLATE_CAP_PER_LEVEL[level]:
        return False, "template_cap"
    if sum(1 for r in existing if r.get("complexity") == level and r.get("template_family") == spec.get("template_family")) >= TEMPLATE_FAMILY_CAP_PER_LEVEL[level]:
        return False, "template_family_cap"
    if not level_kind_ok(spec, existing):
        return False, "kind_cap"
    for k, v in visible_values(spec):
        if sum(1 for r in existing if str((r.get("constraints") or {}).get(k)) == v) >= VISIBLE_VALUE_CAP_TOTAL:
            return False, f"visible_value_cap:{k}:{v}"
    return True, "ok"


def gold_overlap_ok(gold_qids: List[str], existing: List[Dict[str, Any]]) -> Tuple[bool, str]:
    s = set(gold_qids)
    if not s:
        return False, "empty_gold"
    for r in existing:
        t = set(r.get("gold_answer_qids") or [])
        if not t:
            continue
        inter = len(s & t)
        union = len(s | t)
        j = inter / union if union else 0.0
        containment = inter / min(len(s), len(t)) if min(len(s), len(t)) else 0.0
        if j >= GOLD_JACCARD_REJECT:
            return False, f"gold_jaccard:{j:.3f}:{r.get('id')}"
        if containment >= GOLD_CONTAINMENT_REJECT and min(len(s), len(t)) >= 5:
            return False, f"gold_containment:{containment:.3f}:{r.get('id')}"
    return True, "ok"


def gold_quality_ok(labels_en: List[str], spec: Dict[str, Any]) -> Tuple[bool, str]:
    norms = [re.sub(r"[\W_]+", "", norm_text(x)) for x in labels_en]
    if len(set(norms)) < len(norms):
        return False, "duplicate_normalized_labels"
    # Avoid very low-value serial-only answer sets such as Starlink-xxxxx or Umbra 01..10 when possible.
    prefix_counts = Counter(re.sub(r"[\s\-–—_]*\d+.*$", "", x).strip().lower() for x in labels_en)
    if prefix_counts and prefix_counts.most_common(1)[0][1] >= max(6, int(0.75 * len(labels_en))):
        if spec["level"] in {"L1", "L2"}:
            return False, "too_serial_for_low_level"
    return True, "ok"


def build_record_from_spec(spec: Dict[str, Any], existing: List[Dict[str, Any]]) -> Tuple[Optional[Dict[str, Any]], Dict[str, Any]]:
    level = spec["level"]
    requested = int(spec.get("requested_count") or REQUESTED_BY_LEVEL[level])
    min_gold = MIN_GOLD_BY_LEVEL[level]
    max_gold = MAX_GOLD_BY_LEVEL[level]
    select_query, _, _ = release_gold_queries(spec)
    debug = {"template_id": spec.get("template_id"), "constraints": spec.get("constraints")}

    try:
        data = wdqs(select_query)
    except Exception as e:
        return None, {**debug, "reject_reason": "wdqs_error", "error": str(e)[:300]}

    rows = data.get("results", {}).get("bindings", [])
    if len(rows) >= 701:
        return None, {**debug, "reject_reason": "wdqs_limit_hit_or_too_broad:701"}

    # One SELECT with optional labels gives both universes:
    # all matching QIDs and the subset with English labels.
    all_qids_seen: Set[str] = set()
    gold_qids, labels_en, labels_ru = [], [], []
    missing_en_qids = []
    seen_en = set()
    for b in rows:
        qid = wd_qid_from_uri(wd_value(b, "item"))
        if not qid:
            continue
        all_qids_seen.add(qid)
        en = clean_label(wd_value(b, "itemLabelEn"))
        ru = clean_label(wd_value(b, "itemLabelRu") or en)
        if not en:
            missing_en_qids.append(qid)
            continue
        if qid in seen_en:
            continue
        seen_en.add(qid)
        gold_qids.append(qid)
        labels_en.append(en)
        labels_ru.append(ru or en)

    count_no_label = len(all_qids_seen)
    count_label = len(gold_qids)
    debug["complete_count_with_en_label"] = count_label
    debug["diagnostic_count_without_label_filter"] = count_no_label

    if count_label < min_gold:
        return None, {**debug, "reject_reason": f"too_few_gold:{count_label}"}
    if count_label > max_gold:
        return None, {**debug, "reject_reason": f"too_many_gold:{count_label}"}
    if MUSIC_STRICT_EN_LABEL_COMPLETENESS and count_no_label != count_label:
        return None, {**debug, "reject_reason": f"unlabeled_extra:{count_label}/{count_no_label}", "missing_en_qids_sample": missing_en_qids[:10]}
    if len(gold_qids) < requested:
        return None, {**debug, "reject_reason": f"less_than_requested:{len(gold_qids)}"}

    ok, reason = gold_quality_ok(labels_en, spec)
    if not ok:
        return None, {**debug, "reject_reason": reason}
    ok, reason = gold_overlap_ok(gold_qids, existing)
    if not ok:
        return None, {**debug, "reject_reason": reason}

    created_at = _dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
    record = {
        "id": "__PENDING_ID__",
        "domain": MUSIC_DOMAIN_NAME,
        "complexity": level,
        "query_text_ru": spec["query_text_ru"],
        "constraints": spec["constraints"],
        "requested_count": requested,
        "gold_answer_qids": gold_qids,
        "gold_answer_labels_ru": labels_ru,
        "sparql_query": select_query.strip(),
        "created_at": created_at,
        "query_text_en": spec["query_text_en"],
        "gold_answer_labels_en": labels_en,
        "is_advanced": level in {"L3", "L4", "L5"},
        "template_id": spec["template_id"],
        "template_family": spec["template_family"],
        "gold_truncated": False,
        "ask_validator_sparql": ask_query_for_spec(spec),
        "local_validator": {
            "type": "none_wdqs_only",
            "source": "Wikidata Query Service",
            "applies_after": "ask_validator_sparql",
            "filters": spec["constraints"],
            "label_matching_used": False,
            "note": "All constraints for this music task are represented directly in the WDQS ASK validator; no external local validator is required.",
        },
        "gold_collection_meta": {
            "source": "wikidata_sparql",
            "wdqs_candidate_limit": 701,
            "rows_returned_by_wdqs": len(gold_qids),
            "gold_returned_before_limits": len(gold_qids),
            "dropped_no_qid_count": 0,
            "dropped_no_en_label_count": 0,
            "label_sources": {"ru_label": sum(1 for ru,en in zip(labels_ru, labels_en) if ru != en), "en_fallback_for_ru": sum(1 for ru,en in zip(labels_ru, labels_en) if ru == en)},
            "gold_may_be_incomplete_due_to_wdqs_limit": False,
            "quality_filter_applied": True,
            "constraints_are_wdqs_only": True,
            "constraints": spec["constraints"],
            "gold_limit": 300,
            "gold_returned": len(gold_qids),
            "gold_total_before_limit": len(gold_qids),
            "complete_count_with_en_label": count_label,
            "diagnostic_count_without_label_filter": count_no_label,
            "gold_truncated_by_local_limit": False,
            "template_id": spec["template_id"],
            "template_family": spec["template_family"],
            "patch_version": MUSIC_GENERATOR_VERSION,
            "answer_kind": spec["answer_kind"],
            "answer_kind_quota_bucket": spec["answer_kind_quota_bucket"],
            "bridge_meta": spec.get("bridge_meta", {}),
            "candidate_constraints_signature": constraints_signature(spec["constraints"]),
        },
        "gold_answer_imdb_ids": [],
        "gold_answer_imdb_titles": [],
    }
    return record, {**debug, "accepted_gold": len(gold_qids)}


In [6]:
# -------------------------
# NLG and candidate queue construction
# -------------------------
def release_query_text(kind: str, constraints: Dict[str, Any], requested: int) -> Tuple[str, str]:
    kind_ru = RELEASE_KINDS[kind]["ru"]
    kind_en = RELEASE_KINDS[kind]["en_plural"]
    parts_ru, parts_en = [], []
    if "genre" in constraints:
        parts_ru.append(f"жанра «{constraints['genre_ru']}»")
        parts_en.append(f"in the genre {constraints['genre']}")
    if "record_label" in constraints:
        parts_ru.append(f"выпущенных лейблом «{constraints['record_label']}»")
        parts_en.append(f"released by {constraints['record_label']}")
    if "language" in constraints:
        parts_ru.append(f"на языке «{constraints.get('language_ru', constraints['language'])}»")
        parts_en.append(f"whose language is {constraints['language']}")
    if "performer_country" in constraints:
        parts_ru.append(f"у исполнителей из страны «{constraints.get('performer_country_ru', constraints['performer_country'])}»")
        parts_en.append(f"by performers from {constraints['performer_country']}")
    if "performer_genre" in constraints:
        parts_ru.append(f"у исполнителей жанра «{constraints.get('performer_genre_ru', constraints['performer_genre'])}»")
        parts_en.append(f"by performers associated with {constraints['performer_genre']}")
    if "same_record_label_as_release" in constraints:
        parts_ru.append(f"выпущенных тем же лейблом, что и «{constraints['same_record_label_as_release']}»")
        parts_en.append(f"released by the same record label as {constraints['same_record_label_as_release']}")
    if "performer_has_release_kind" in constraints:
        yr = ""
        if "performer_has_release_year_min" in constraints:
            yr = f" в {constraints['performer_has_release_year_min']}–{constraints['performer_has_release_year_max']} годах"
        parts_ru.append(f"у исполнителей, у которых также есть {constraints['performer_has_release_kind_ru']} жанра «{constraints['performer_has_release_genre_ru']}»{yr}")
        parts_en.append(f"by performers who also have {constraints['performer_has_release_kind_en']} in the genre {constraints['performer_has_release_genre']}{(' between '+str(constraints['performer_has_release_year_min'])+' and '+str(constraints['performer_has_release_year_max'])) if 'performer_has_release_year_min' in constraints else ''}")
    if "publication_year_min" in constraints:
        parts_ru.append(f"опубликованных в {constraints['publication_year_min']}–{constraints['publication_year_max']} годах")
        parts_en.append(f"published between {constraints['publication_year_min']} and {constraints['publication_year_max']}")
    ru = f"Назови {requested} {kind_ru} " + ", ".join(parts_ru) + "."
    en = f"Name {requested} {kind_en} " + ", ".join(parts_en) + "."
    return ru, en


def performer_query_text(conditions: List[Dict[str, Any]], requested: int) -> Tuple[str, str]:
    ru_parts, en_parts = [], []
    for i, c in enumerate(conditions, start=1):
        k_ru = RELEASE_KINDS[c["kind"]]["ru"]
        k_en = RELEASE_KINDS[c["kind"]]["en_plural"]
        yr_ru = f" в {c['year_min']}–{c['year_max']} годах" if c.get("year_min") else ""
        yr_en = f" between {c['year_min']} and {c['year_max']}" if c.get("year_min") else ""
        ru_parts.append(f"есть {k_ru} жанра «{c['genre_ru']}»{yr_ru}")
        en_parts.append(f"have {k_en} in the genre {c['genre']}{yr_en}")
    ru = f"Назови {requested} исполнителей, у которых " + " и ".join(ru_parts) + "."
    en = f"Name {requested} performers who " + " and ".join(en_parts) + "."
    return ru, en


def release_body_base(kind: str, period: Optional[Dict[str, int]] = None) -> List[str]:
    lines = [release_kind_pattern("?item", kind)]
    if period:
        lines.append(date_filter("?item", int(period["from"]), int(period["to"]), "?pub_date"))
    return lines


def add_genre(lines: List[str], genre: Dict[str, Any]) -> None:
    lines.append(f"?item wdt:P136 {q(genre['qid'])} .")


def add_label(lines: List[str], label: Dict[str, Any]) -> None:
    lines.append(f"?item wdt:P264 {q(label['qid'])} .")


def add_language(lines: List[str], lang: Dict[str, Any]) -> None:
    lines.append(f"?item wdt:P407 {q(lang['qid'])} .")


def add_performer_country(lines: List[str], country: Dict[str, Any]) -> None:
    lines.append("?item wdt:P175 ?performer .")
    lines.append(f"{{ ?performer wdt:P27 {q(country['qid'])} . }} UNION {{ ?performer wdt:P495 {q(country['qid'])} . }} UNION {{ ?performer wdt:P17 {q(country['qid'])} . }}")


def add_performer_genre(lines: List[str], genre: Dict[str, Any]) -> None:
    lines.append("?item wdt:P175 ?performer .")
    lines.append(f"?performer wdt:P136 {q(genre['qid'])} .")


def add_same_label_as_seed(lines: List[str], seed: Dict[str, Any]) -> None:
    lines.append(f"{q(seed['qid'])} wdt:P264 ?bridge_label .")
    lines.append("?item wdt:P264 ?bridge_label .")
    lines.append(f"FILTER(?item != {q(seed['qid'])})")


def add_performer_has_other_release(lines: List[str], cond: Dict[str, Any]) -> None:
    lines.append("?item wdt:P175 ?bridge_performer .")
    lines.append("?other_release wdt:P175 ?bridge_performer .")
    lines.append(release_kind_pattern("?other_release", cond["kind"]))
    lines.append(f"?other_release wdt:P136 {q(cond['genre_qid'])} .")
    if cond.get("year_min"):
        lines.append(date_filter("?other_release", int(cond["year_min"]), int(cond["year_max"]), "?other_pub_date"))
    lines.append("FILTER(?other_release != ?item)")


def make_release_spec(level: str, kind: str, template_id: str, template_family: str, constraints_extra: Dict[str, Any], body_lines: List[str], bridge_meta: Dict[str, Any]) -> Dict[str, Any]:
    requested = REQUESTED_BY_LEVEL[level]
    query_constraints = {"kind": kind, **constraints_extra}
    query_ru, query_en = release_query_text(kind, query_constraints, requested)
    # constraints_extra does not include kind here; release_answer_spec adds it.
    return release_answer_spec(level, kind, template_id, template_family, body_lines, constraints_extra, query_ru, query_en, bridge_meta)


def make_performer_condition(kind: str, genre: Dict[str, Any], period: Optional[Dict[str, int]] = None) -> Dict[str, Any]:
    d = {"kind": kind, "genre": genre["label_en"], "genre_ru": genre.get("label_ru") or genre["label_en"], "genre_qid": genre["qid"]}
    if period:
        d.update({"year_min": int(period["from"]), "year_max": int(period["to"])})
    return d


def make_performer_spec(level: str, template_id: str, template_family: str, conditions: List[Dict[str, Any]]) -> Dict[str, Any]:
    body = []
    bridge_meta = {"conditions": []}
    constraints = {}
    for idx, cond in enumerate(conditions, start=1):
        rel = f"?rel{idx}"
        body.append(f"{rel} wdt:P175 ?item .")
        body.append(release_kind_pattern(rel, cond["kind"]))
        body.append(f"{rel} wdt:P136 {q(cond['genre_qid'])} .")
        if cond.get("year_min"):
            body.append(date_filter(rel, int(cond["year_min"]), int(cond["year_max"]), f"?rel{idx}_date"))
        constraints[f"released_kind_{idx}"] = cond["kind"]
        constraints[f"released_genre_{idx}"] = cond["genre"]
        constraints[f"released_genre_ru_{idx}"] = cond["genre_ru"]
        if cond.get("year_min"):
            constraints[f"released_publication_year_min_{idx}"] = int(cond["year_min"])
            constraints[f"released_publication_year_max_{idx}"] = int(cond["year_max"])
        bridge_meta["conditions"].append({k: v for k, v in cond.items() if k != "genre_qid"} | {"genre_qid": cond["genre_qid"]})
    for i in range(1, len(conditions)+1):
        for j in range(i+1, len(conditions)+1):
            body.append(f"FILTER(?rel{i} != ?rel{j})")
    ru, en = performer_query_text(conditions, REQUESTED_BY_LEVEL[level])
    return performer_answer_spec(level, template_id, template_family, body, constraints, ru, en, bridge_meta)


def build_candidate_queue(cache: Dict[str, Any]) -> List[Dict[str, Any]]:
    specs: List[Dict[str, Any]] = []
    # L1: release kind + one meaningful facet + period.
    for kind in ["studio album", "EP", "single"]:
        for period in PERIODS:
            genres = get_facets(cache, kind, period, "genre", CURATED_GENRES)[:16]
            labels = get_facets(cache, kind, period, "record_label", CURATED_RECORD_LABELS)[:16]
            languages = get_facets(cache, kind, period, "language", CURATED_LANGUAGES)[:10]
            for genre in genres:
                lines = release_body_base(kind, period); add_genre(lines, genre)
                extra = {"genre": genre["label_en"], "genre_ru": genre.get("label_ru") or genre["label_en"], **period_constraint(period)}
                specs.append(make_release_spec("L1", kind, "music_l1_kind_genre_period", "kind_genre_period", extra, lines, {"genre_qid": genre["qid"]}))
            for label in labels:
                lines = release_body_base(kind, period); add_label(lines, label)
                extra = {"record_label": label["label_en"], **period_constraint(period)}
                specs.append(make_release_spec("L1", kind, "music_l1_kind_label_period", "kind_label_period", extra, lines, {"record_label_qid": label["qid"]}))
            for lang in languages:
                lines = release_body_base(kind, period); add_language(lines, lang)
                extra = {"language": lang["label_en"], "language_ru": lang.get("label_ru") or lang["label_en"], **period_constraint(period)}
                specs.append(make_release_spec("L1", kind, "music_l1_kind_language_period", "kind_language_period", extra, lines, {"language_qid": lang["qid"]}))

    # L2: release kind + two facets, often with performer bridge.
    for kind in ["studio album", "EP", "single"]:
        for period in PERIODS:
            genres = get_facets(cache, kind, period, "genre", CURATED_GENRES)[:12]
            labels = get_facets(cache, kind, period, "record_label", CURATED_RECORD_LABELS)[:12]
            languages = get_facets(cache, kind, period, "language", CURATED_LANGUAGES)[:8]
            countries = get_facets(cache, kind, period, "performer_country", CURATED_COUNTRIES)[:10]
            for genre in genres[:8]:
                for label in labels[:5]:
                    lines = release_body_base(kind, period); add_genre(lines, genre); add_label(lines, label)
                    extra = {"genre": genre["label_en"], "genre_ru": genre.get("label_ru") or genre["label_en"], "record_label": label["label_en"], **period_constraint(period)}
                    specs.append(make_release_spec("L2", kind, "music_l2_kind_genre_label_period", "kind_genre_label_period", extra, lines, {"genre_qid": genre["qid"], "record_label_qid": label["qid"]}))
                for lang in languages[:4]:
                    lines = release_body_base(kind, period); add_genre(lines, genre); add_language(lines, lang)
                    extra = {"genre": genre["label_en"], "genre_ru": genre.get("label_ru") or genre["label_en"], "language": lang["label_en"], "language_ru": lang.get("label_ru") or lang["label_en"], **period_constraint(period)}
                    specs.append(make_release_spec("L2", kind, "music_l2_kind_genre_language_period", "kind_genre_language_period", extra, lines, {"genre_qid": genre["qid"], "language_qid": lang["qid"]}))
                for country in countries[:4]:
                    lines = release_body_base(kind, period); add_genre(lines, genre); add_performer_country(lines, country)
                    extra = {"genre": genre["label_en"], "genre_ru": genre.get("label_ru") or genre["label_en"], "performer_country": country["label_en"], "performer_country_ru": country.get("label_ru") or country["label_en"], **period_constraint(period)}
                    specs.append(make_release_spec("L2", kind, "music_l2_kind_genre_performer_country_period", "kind_genre_performer_country_period", extra, lines, {"genre_qid": genre["qid"], "performer_country_qid": country["qid"]}))


            # Extra high-recall L2 patterns that do not require discovery.
            for label in labels[:6]:
                for country in countries[:5]:
                    lines = release_body_base(kind, period); add_label(lines, label); add_performer_country(lines, country)
                    extra = {"record_label": label["label_en"], "performer_country": country["label_en"], "performer_country_ru": country.get("label_ru") or country["label_en"], **period_constraint(period)}
                    specs.append(make_release_spec("L2", kind, "music_l2_kind_label_performer_country_period", "kind_label_performer_country_period", extra, lines, {"record_label_qid": label["qid"], "performer_country_qid": country["qid"]}))
            for label in labels[:5]:
                for lang in languages[:4]:
                    lines = release_body_base(kind, period); add_label(lines, label); add_language(lines, lang)
                    extra = {"record_label": label["label_en"], "language": lang["label_en"], "language_ru": lang.get("label_ru") or lang["label_en"], **period_constraint(period)}
                    specs.append(make_release_spec("L2", kind, "music_l2_kind_label_language_period", "kind_label_language_period", extra, lines, {"record_label_qid": label["qid"], "language_qid": lang["qid"]}))

    # L3: explicit multihop via performer genre / same label as seed / performer answer with two release conditions.
    for kind in ["studio album", "EP", "single"]:
        for period in PERIODS:
            genres = get_facets(cache, kind, period, "genre", CURATED_GENRES)[:10]
            performer_genres = get_facets(cache, kind, period, "performer_genre", CURATED_GENRES)[:10]
            for genre in genres[:7]:
                for pgenre in performer_genres[:4]:
                    if pgenre["qid"] == genre["qid"]:
                        continue
                    lines = release_body_base(kind, period); add_genre(lines, genre); add_performer_genre(lines, pgenre)
                    extra = {"genre": genre["label_en"], "genre_ru": genre.get("label_ru") or genre["label_en"], "performer_genre": pgenre["label_en"], "performer_genre_ru": pgenre.get("label_ru") or pgenre["label_en"], **period_constraint(period)}
                    specs.append(make_release_spec("L3", kind, "music_l3_kind_genre_performer_genre_period", "performer_genre_bridge", extra, lines, {"genre_qid": genre["qid"], "performer_genre_qid": pgenre["qid"]}))
    for seed in (cache.get("seeds") or [])[:100]:
        for kind in ["studio album", "EP"]:
            lines = release_body_base(kind, None); add_same_label_as_seed(lines, seed)
            extra = {"same_record_label_as_release": seed["label_en"], "same_record_label_as_release_ru": seed.get("label_ru") or seed["label_en"], "record_label": seed.get("record_label_en")}
            specs.append(make_release_spec("L3", kind, "music_l3_same_label_as_seed", "same_label_bridge", extra, lines, {"seed_release_qid": seed["qid"], "record_label_qid": seed.get("record_label_qid")}))
    # Performer answer L3.
    for period_a, period_b in zip(PERIODS[:-1], PERIODS[1:]):
        for g1 in CURATED_GENRES[:8]:
            for g2 in CURATED_GENRES[2:10]:
                if g1["qid"] == g2["qid"]:
                    continue
                conds = [make_performer_condition("studio album", g1, period_a), make_performer_condition("single", g2, period_b)]
                specs.append(make_performer_spec("L3", "music_l3_performer_two_release_conditions", "performer_evidence_intersection", conds))

    # L4: same label + genre + period; performer has other evidence release.
    for seed in (cache.get("seeds") or [])[:100]:
        genre = {"qid": seed.get("genre_qid"), "label_en": seed.get("genre_en"), "label_ru": seed.get("genre_ru")}
        if not genre.get("qid"):
            continue
        for period in PERIODS:
            lines = release_body_base("studio album", period); add_same_label_as_seed(lines, seed); add_genre(lines, genre)
            extra = {"same_record_label_as_release": seed["label_en"], "same_record_label_as_release_ru": seed.get("label_ru") or seed["label_en"], "record_label": seed.get("record_label_en"), "genre": genre["label_en"], "genre_ru": genre.get("label_ru") or genre["label_en"], **period_constraint(period)}
            specs.append(make_release_spec("L4", "studio album", "music_l4_same_label_genre_period", "same_label_genre_bridge", extra, lines, {"seed_release_qid": seed["qid"], "record_label_qid": seed.get("record_label_qid"), "genre_qid": genre["qid"]}))
    for kind in ["studio album", "EP", "single"]:
        for period in PERIODS:
            genres = get_facets(cache, kind, period, "genre", CURATED_GENRES)[:8]
            for genre in genres[:5]:
                for other_genre in CURATED_GENRES[:6]:
                    if genre["qid"] == other_genre["qid"]:
                        continue
                    lines = release_body_base(kind, period); add_genre(lines, genre)
                    other_cond = {"kind": "studio album" if kind != "studio album" else "single", "genre_qid": other_genre["qid"], "genre": other_genre["label_en"], "genre_ru": other_genre.get("label_ru") or other_genre["label_en"], "year_min": 1981, "year_max": 2025}
                    add_performer_has_other_release(lines, other_cond)
                    extra = {"genre": genre["label_en"], "genre_ru": genre.get("label_ru") or genre["label_en"], "performer_has_release_kind": other_cond["kind"], "performer_has_release_kind_ru": RELEASE_KINDS[other_cond["kind"]]["ru"], "performer_has_release_kind_en": RELEASE_KINDS[other_cond["kind"]]["en_plural"], "performer_has_release_genre": other_cond["genre"], "performer_has_release_genre_ru": other_cond["genre_ru"], "performer_has_release_year_min": other_cond["year_min"], "performer_has_release_year_max": other_cond["year_max"], **period_constraint(period)}
                    specs.append(make_release_spec("L4", kind, "music_l4_release_performer_other_release", "performer_other_release_bridge", extra, lines, {"genre_qid": genre["qid"], "other_release_condition": other_cond}))
    # Performer answer L4: two strong conditions, both with periods.
    for g1 in CURATED_GENRES[:8]:
        for g2 in CURATED_GENRES[3:11]:
            if g1["qid"] == g2["qid"]:
                continue
            conds = [make_performer_condition("studio album", g1, PERIODS[1]), make_performer_condition("EP", g2, PERIODS[2])]
            specs.append(make_performer_spec("L4", "music_l4_performer_album_ep_conditions", "performer_evidence_intersection", conds))


    # L4: release label + genre + performer-other-release bridge, no seed discovery needed.
    for kind in ["studio album", "EP", "single"]:
        for period in PERIODS:
            labels = get_facets(cache, kind, period, "record_label", CURATED_RECORD_LABELS)[:8]
            genres = get_facets(cache, kind, period, "genre", CURATED_GENRES)[:8]
            for label in labels[:5]:
                for genre in genres[:5]:
                    for other_genre in CURATED_GENRES[:5]:
                        if other_genre["qid"] == genre["qid"]:
                            continue
                        lines = release_body_base(kind, period); add_label(lines, label); add_genre(lines, genre)
                        other_cond = {"kind": "single" if kind != "single" else "studio album", "genre_qid": other_genre["qid"], "genre": other_genre["label_en"], "genre_ru": other_genre.get("label_ru") or other_genre["label_en"], "year_min": 1981, "year_max": 2025}
                        add_performer_has_other_release(lines, other_cond)
                        extra = {"record_label": label["label_en"], "genre": genre["label_en"], "genre_ru": genre.get("label_ru") or genre["label_en"], "performer_has_release_kind": other_cond["kind"], "performer_has_release_kind_ru": RELEASE_KINDS[other_cond["kind"]]["ru"], "performer_has_release_kind_en": RELEASE_KINDS[other_cond["kind"]]["en_plural"], "performer_has_release_genre": other_cond["genre"], "performer_has_release_genre_ru": other_cond["genre_ru"], "performer_has_release_year_min": other_cond["year_min"], "performer_has_release_year_max": other_cond["year_max"], **period_constraint(period)}
                        specs.append(make_release_spec("L4", kind, "music_l4_label_genre_performer_other", "label_genre_performer_other_bridge", extra, lines, {"record_label_qid": label["qid"], "genre_qid": genre["qid"], "other_release_condition": other_cond}))

    # L5: three-condition performer intersections + release with same-label and performer-other-release bridge.
    for g1 in CURATED_GENRES[:7]:
        for g2 in CURATED_GENRES[2:8]:
            for g3 in CURATED_GENRES[5:11]:
                if len({g1["qid"], g2["qid"], g3["qid"]}) < 3:
                    continue
                conds = [
                    make_performer_condition("studio album", g1, PERIODS[0]),
                    make_performer_condition("single", g2, PERIODS[2]),
                    make_performer_condition("EP", g3, PERIODS[3]),
                ]
                specs.append(make_performer_spec("L5", "music_l5_performer_three_release_conditions", "performer_three_evidence_intersection", conds))
    for seed in (cache.get("seeds") or [])[:100]:
        genre = {"qid": seed.get("genre_qid"), "label_en": seed.get("genre_en"), "label_ru": seed.get("genre_ru")}
        if not genre.get("qid"):
            continue
        for other_genre in CURATED_GENRES[:6]:
            if other_genre["qid"] == genre["qid"]:
                continue
            lines = release_body_base("studio album", PERIODS[2]); add_same_label_as_seed(lines, seed); add_genre(lines, genre)
            other_cond = {"kind": "single", "genre_qid": other_genre["qid"], "genre": other_genre["label_en"], "genre_ru": other_genre.get("label_ru") or other_genre["label_en"], "year_min": 1981, "year_max": 2025}
            add_performer_has_other_release(lines, other_cond)
            extra = {"same_record_label_as_release": seed["label_en"], "same_record_label_as_release_ru": seed.get("label_ru") or seed["label_en"], "record_label": seed.get("record_label_en"), "genre": genre["label_en"], "genre_ru": genre.get("label_ru") or genre["label_en"], "performer_has_release_kind": "single", "performer_has_release_kind_ru": RELEASE_KINDS["single"]["ru"], "performer_has_release_kind_en": RELEASE_KINDS["single"]["en_plural"], "performer_has_release_genre": other_cond["genre"], "performer_has_release_genre_ru": other_cond["genre_ru"], "performer_has_release_year_min": other_cond["year_min"], "performer_has_release_year_max": other_cond["year_max"], **period_constraint(PERIODS[2])}
            specs.append(make_release_spec("L5", "studio album", "music_l5_same_label_genre_performer_other", "same_label_and_performer_other_bridge", extra, lines, {"seed_release_qid": seed["qid"], "record_label_qid": seed.get("record_label_qid"), "genre_qid": genre["qid"], "other_release_condition": other_cond}))

    # L5: label + genre + performer country + performer-other-release bridge, no seed discovery needed.
    for kind in ["studio album", "EP", "single"]:
        for period in PERIODS[1:]:
            labels = get_facets(cache, kind, period, "record_label", CURATED_RECORD_LABELS)[:8]
            countries = get_facets(cache, kind, period, "performer_country", CURATED_COUNTRIES)[:8]
            genres = get_facets(cache, kind, period, "genre", CURATED_GENRES)[:8]
            for label in labels[:4]:
                for country in countries[:4]:
                    for genre in genres[:4]:
                        for other_genre in CURATED_GENRES[4:8]:
                            if other_genre["qid"] == genre["qid"]:
                                continue
                            lines = release_body_base(kind, period); add_label(lines, label); add_performer_country(lines, country); add_genre(lines, genre)
                            other_cond = {"kind": "single" if kind != "single" else "studio album", "genre_qid": other_genre["qid"], "genre": other_genre["label_en"], "genre_ru": other_genre.get("label_ru") or other_genre["label_en"], "year_min": 1981, "year_max": 2025}
                            add_performer_has_other_release(lines, other_cond)
                            extra = {"record_label": label["label_en"], "performer_country": country["label_en"], "performer_country_ru": country.get("label_ru") or country["label_en"], "genre": genre["label_en"], "genre_ru": genre.get("label_ru") or genre["label_en"], "performer_has_release_kind": other_cond["kind"], "performer_has_release_kind_ru": RELEASE_KINDS[other_cond["kind"]]["ru"], "performer_has_release_kind_en": RELEASE_KINDS[other_cond["kind"]]["en_plural"], "performer_has_release_genre": other_cond["genre"], "performer_has_release_genre_ru": other_cond["genre_ru"], "performer_has_release_year_min": other_cond["year_min"], "performer_has_release_year_max": other_cond["year_max"], **period_constraint(period)}
                            specs.append(make_release_spec("L5", kind, "music_l5_label_country_genre_performer_other", "label_country_genre_performer_other_bridge", extra, lines, {"record_label_qid": label["qid"], "performer_country_qid": country["qid"], "genre_qid": genre["qid"], "other_release_condition": other_cond}))

    # Stable shuffle by hash to interleave templates/facets.
    
    def _spec_priority(s):
        tpl = s.get("template_id", "")
        pri = 0
        if "label" in tpl:
            pri -= 200
        if "performer_country" in tpl:
            pri -= 80
        if "performer_other" in tpl:
            pri -= 120
        if "language" in tpl and s.get("answer_kind_quota_bucket") == "studio album":
            pri += 80
        if tpl == "music_l1_kind_genre_period" and s.get("answer_kind_quota_bucket") == "studio album":
            pri += 120
        return (s["level"], pri, spec_hash(s) % 1000003)
    specs = sorted(specs, key=_spec_priority)
    print("candidate specs:", len(specs), dict(Counter(s["level"] for s in specs)))
    print("candidate templates top:", Counter(s["template_id"] for s in specs).most_common(20))
    return specs


# --- v36 override: compact, high-yield candidate queue, records first ---
def _facet_by_label(rows: List[Dict[str, Any]], label: str) -> Optional[Dict[str, Any]]:
    label_n = norm_text(label)
    for r in rows:
        if norm_text(r.get("label_en")) == label_n or norm_text(r.get("label_ru")) == label_n:
            return r
    return None


def _set_priority(spec: Dict[str, Any], priority: int) -> Dict[str, Any]:
    spec["priority"] = int(priority)
    return spec


def _add_release_spec(out: List[Dict[str, Any]], *, priority: int, level: str, kind: str, template_id: str, template_family: str,
                      period: Optional[Dict[str, int]], genre: Optional[Dict[str, Any]] = None,
                      label: Optional[Dict[str, Any]] = None, language: Optional[Dict[str, Any]] = None,
                      performer_country: Optional[Dict[str, Any]] = None, performer_genre: Optional[Dict[str, Any]] = None,
                      other_release: Optional[Dict[str, Any]] = None) -> None:
    lines = release_body_base(kind, period)
    extra: Dict[str, Any] = {}
    bridge: Dict[str, Any] = {}
    if genre:
        add_genre(lines, genre)
        extra.update({"genre": genre["label_en"], "genre_ru": genre.get("label_ru") or genre["label_en"]})
        bridge["genre_qid"] = genre["qid"]
    if label:
        add_label(lines, label)
        extra.update({"record_label": label["label_en"]})
        bridge["record_label_qid"] = label["qid"]
    if language:
        add_language(lines, language)
        extra.update({"language": language["label_en"], "language_ru": language.get("label_ru") or language["label_en"]})
        bridge["language_qid"] = language["qid"]
    if performer_country:
        add_performer_country(lines, performer_country)
        extra.update({"performer_country": performer_country["label_en"], "performer_country_ru": performer_country.get("label_ru") or performer_country["label_en"]})
        bridge["performer_country_qid"] = performer_country["qid"]
    if performer_genre:
        add_performer_genre(lines, performer_genre)
        extra.update({"performer_genre": performer_genre["label_en"], "performer_genre_ru": performer_genre.get("label_ru") or performer_genre["label_en"]})
        bridge["performer_genre_qid"] = performer_genre["qid"]
    if other_release:
        add_performer_has_other_release(lines, other_release)
        extra.update({
            "performer_has_release_kind": other_release["kind"],
            "performer_has_release_kind_ru": RELEASE_KINDS[other_release["kind"]]["ru"],
            "performer_has_release_kind_en": RELEASE_KINDS[other_release["kind"]]["en_plural"],
            "performer_has_release_genre": other_release["genre"],
            "performer_has_release_genre_ru": other_release["genre_ru"],
            "performer_has_release_year_min": other_release["year_min"],
            "performer_has_release_year_max": other_release["year_max"],
        })
        bridge["other_release_condition"] = other_release
    if period:
        extra.update(period_constraint(period))
    spec = make_release_spec(level, kind, template_id, template_family, extra, lines, bridge)
    out.append(_set_priority(spec, priority))


def build_candidate_queue(cache: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Build a compact prioritized queue that starts producing benchmark records quickly.

    v35 built 6k+ candidates and showed progress over candidate attempts. v36 builds a smaller,
    ordered queue by complexity, with known high-yield patterns first. Golds are still checked in WDQS.
    """
    specs: List[Dict[str, Any]] = []
    genres = CURATED_GENRES
    labels = CURATED_RECORD_LABELS
    languages = CURATED_LANGUAGES
    countries = CURATED_COUNTRIES

    # Useful named facets for early high-yield records.
    G = {g["label_en"]: g for g in genres}
    L = {l["label_en"]: l for l in labels}
    Lang = {l["label_en"]: l for l in languages}
    C = {c["label_en"]: c for c in countries}

    # L1: still simple, but not too broad. Avoid pure genre-only album queries.
    l1_blueprints = [
        ("studio album", PERIODS[0], L.get("Blue Note"), G.get("jazz"), None, None),
        ("studio album", PERIODS[0], L.get("Motown"), G.get("soul music"), None, None),
        ("studio album", PERIODS[1], L.get("Def Jam Recordings"), G.get("hip hop music"), None, None),
        ("studio album", PERIODS[2], L.get("XL Recordings"), G.get("electronic music"), None, None),
        ("studio album", PERIODS[0], L.get("Atlantic Records"), G.get("rhythm and blues"), None, None),
        ("studio album", PERIODS[1], L.get("Capitol Records"), G.get("rock music"), None, None),
        ("EP", PERIODS[2], None, G.get("rock music"), None, C.get("United States")),
        ("EP", PERIODS[3], None, G.get("pop music"), Lang.get("English"), None),
        ("EP", PERIODS[3], None, G.get("hip hop music"), None, C.get("United States")),
        ("single", PERIODS[2], None, G.get("rock music"), Lang.get("English"), None),
        ("single", PERIODS[3], None, G.get("Korean"), None, None),
        ("single", PERIODS[3], None, G.get("pop music"), None, C.get("Japan")),
    ]
    for kind, period, label, genre, lang, country in l1_blueprints:
        if not genre: continue
        _add_release_spec(specs, priority=10, level="L1", kind=kind, template_id="music_l1_curated_label_genre_period" if label else "music_l1_curated_genre_bridge_period", template_family="curated_release_period", period=period, genre=genre, label=label, language=lang, performer_country=country)

    # Extra L1 fallback: genre + language/country for EP/single; label+period for albums.
    for period in PERIODS:
        for kind in ["EP", "single"]:
            for genre in genres[:8]:
                _add_release_spec(specs, priority=40, level="L1", kind=kind, template_id="music_l1_kind_genre_period", template_family="kind_genre_period", period=period, genre=genre)
        for label in labels[:8]:
            _add_release_spec(specs, priority=45, level="L1", kind="studio album", template_id="music_l1_album_label_period", template_family="album_label_period", period=period, label=label)

    # L2: release kind + two facets + period.
    for period in PERIODS:
        for kind in ["studio album", "EP", "single"]:
            for genre in genres[:8]:
                for label in labels[:5]:
                    _add_release_spec(specs, priority=100, level="L2", kind=kind, template_id="music_l2_kind_genre_label_period", template_family="kind_genre_label_period", period=period, genre=genre, label=label)
                for country in countries[:6]:
                    _add_release_spec(specs, priority=115, level="L2", kind=kind, template_id="music_l2_kind_genre_performer_country_period", template_family="kind_genre_country_period", period=period, genre=genre, performer_country=country)
                for lang in languages[:5]:
                    _add_release_spec(specs, priority=130, level="L2", kind=kind, template_id="music_l2_kind_genre_language_period", template_family="kind_genre_language_period", period=period, genre=genre, language=lang)

    # L3: explicit performer-genre bridge + country/label bridge.
    for period in PERIODS:
        for kind in ["studio album", "EP", "single"]:
            for genre in genres[:7]:
                for pgenre in genres[:7]:
                    if pgenre["qid"] == genre["qid"]:
                        continue
                    _add_release_spec(specs, priority=200, level="L3", kind=kind, template_id="music_l3_release_genre_performer_genre_period", template_family="performer_genre_bridge", period=period, genre=genre, performer_genre=pgenre)
            for label in labels[:5]:
                for country in countries[:5]:
                    _add_release_spec(specs, priority=215, level="L3", kind=kind, template_id="music_l3_label_country_period", template_family="label_country_bridge", period=period, label=label, performer_country=country)

    # Performer-answer L3: two release-evidence conditions.
    for p1, p2 in [(PERIODS[0], PERIODS[2]), (PERIODS[1], PERIODS[3])]:
        for g1 in genres[:7]:
            for g2 in genres[3:10]:
                if g1["qid"] == g2["qid"]:
                    continue
                conds = [make_performer_condition("studio album", g1, p1), make_performer_condition("single", g2, p2)]
                specs.append(_set_priority(make_performer_spec("L3", "music_l3_performer_album_single_conditions", "performer_evidence_intersection", conds), 230))

    # L4: release answer with performer-other-release bridge.
    for period in PERIODS:
        for kind in ["studio album", "EP", "single"]:
            for genre in genres[:6]:
                for label in labels[:5]:
                    for other_genre in genres[4:9]:
                        if other_genre["qid"] == genre["qid"]:
                            continue
                        other_kind = "single" if kind != "single" else "studio album"
                        other_cond = {"kind": other_kind, "genre_qid": other_genre["qid"], "genre": other_genre["label_en"], "genre_ru": other_genre.get("label_ru") or other_genre["label_en"], "year_min": 1981, "year_max": 2025}
                        _add_release_spec(specs, priority=300, level="L4", kind=kind, template_id="music_l4_label_genre_performer_other_period", template_family="label_genre_performer_other_bridge", period=period, genre=genre, label=label, other_release=other_cond)
                for country in countries[:5]:
                    for other_genre in genres[5:10]:
                        if other_genre["qid"] == genre["qid"]:
                            continue
                        other_kind = "single" if kind != "single" else "studio album"
                        other_cond = {"kind": other_kind, "genre_qid": other_genre["qid"], "genre": other_genre["label_en"], "genre_ru": other_genre.get("label_ru") or other_genre["label_en"], "year_min": 1981, "year_max": 2025}
                        _add_release_spec(specs, priority=315, level="L4", kind=kind, template_id="music_l4_country_genre_performer_other_period", template_family="country_genre_performer_other_bridge", period=period, genre=genre, performer_country=country, other_release=other_cond)

    # Performer-answer L4.
    for g1 in genres[:7]:
        for g2 in genres[4:11]:
            if g1["qid"] == g2["qid"]:
                continue
            conds = [make_performer_condition("studio album", g1, PERIODS[1]), make_performer_condition("EP", g2, PERIODS[2])]
            specs.append(_set_priority(make_performer_spec("L4", "music_l4_performer_album_ep_conditions", "performer_evidence_intersection", conds), 330))

    # L5: stronger release bridges and three-condition performer intersections.
    for period in PERIODS[1:]:
        for kind in ["studio album", "EP", "single"]:
            for label in labels[:5]:
                for country in countries[:5]:
                    for genre in genres[:5]:
                        for other_genre in genres[5:10]:
                            if other_genre["qid"] == genre["qid"]:
                                continue
                            other_kind = "single" if kind != "single" else "studio album"
                            other_cond = {"kind": other_kind, "genre_qid": other_genre["qid"], "genre": other_genre["label_en"], "genre_ru": other_genre.get("label_ru") or other_genre["label_en"], "year_min": 1981, "year_max": 2025}
                            _add_release_spec(specs, priority=400, level="L5", kind=kind, template_id="music_l5_label_country_genre_performer_other_period", template_family="label_country_genre_performer_other_bridge", period=period, genre=genre, label=label, performer_country=country, other_release=other_cond)
    for g1 in genres[:7]:
        for g2 in genres[2:8]:
            for g3 in genres[5:11]:
                if len({g1["qid"], g2["qid"], g3["qid"]}) < 3:
                    continue
                conds = [
                    make_performer_condition("studio album", g1, PERIODS[0]),
                    make_performer_condition("single", g2, PERIODS[2]),
                    make_performer_condition("EP", g3, PERIODS[3]),
                ]
                specs.append(_set_priority(make_performer_spec("L5", "music_l5_performer_three_release_conditions", "performer_three_evidence_intersection", conds), 420))

    # Deduplicate same constraints and sort by level/priority/template hash.
    uniq: Dict[str, Dict[str, Any]] = {}
    for s in specs:
        sig = constraints_signature(s.get("constraints", {}))
        if sig not in uniq or int(s.get("priority", 9999)) < int(uniq[sig].get("priority", 9999)):
            uniq[sig] = s
    specs = list(uniq.values())
    level_order = {"L1": 1, "L2": 2, "L3": 3, "L4": 4, "L5": 5}
    specs.sort(key=lambda s: (level_order.get(s.get("level"), 99), int(s.get("priority", 9999)), s.get("template_id", ""), spec_hash(s) % 1000003))
    print("candidate specs in memory:", len(specs), dict(Counter(s["level"] for s in specs)))
    print("candidate templates top:", Counter(s["template_id"] for s in specs).most_common(18))
    print("records will be appended incrementally to:", MUSIC_OUTPUT_PATH.resolve())
    return specs


In [ ]:
# -------------------------
# Generation loop — v36 per-level accepted-record progress
# -------------------------
def next_id_for_level(existing: List[Dict[str, Any]], level: str) -> str:
    n = 1 + sum(1 for r in existing if r.get("complexity") == level)
    return f"music_albums_{level.lower()}_{n:04d}"


def target_done(records: List[Dict[str, Any]]) -> bool:
    c = Counter(r.get("complexity") for r in records)
    return all(c.get(level, 0) >= target for level, target in MUSIC_TARGET_PER_LEVEL.items())


def level_count(records: List[Dict[str, Any]], level: str) -> int:
    return sum(1 for r in records if r.get("complexity") == level)


def generation_summary(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "total": len(records),
        "by_level": dict(Counter(r.get("complexity") for r in records)),
        "by_kind": dict(Counter((r.get("gold_collection_meta") or {}).get("answer_kind_quota_bucket") for r in records)),
        "by_template": dict(Counter(r.get("template_id") for r in records)),
        "gold_min": min([len(r.get("gold_answer_qids", [])) for r in records], default=0),
        "gold_max": max([len(r.get("gold_answer_qids", [])) for r in records], default=0),
    }


def _fresh_audit() -> Dict[str, Any]:
    audit = _json_load(MUSIC_AUDIT_PATH, None)
    if not isinstance(audit, dict) or audit.get("version") != MUSIC_GENERATOR_VERSION:
        return {"version": MUSIC_GENERATOR_VERSION, "events": [], "created_at": _dt.datetime.utcnow().isoformat()+"Z"}
    return audit


def _save_generation_state(audit: Dict[str, Any], records: List[Dict[str, Any]]) -> None:
    audit["summary"] = generation_summary(records)
    audit["last_update"] = _dt.datetime.utcnow().isoformat()+"Z"
    _json_dump(MUSIC_AUDIT_PATH, audit)
    _json_dump(MUSIC_CHECKPOINT_PATH, {"version": MUSIC_GENERATOR_VERSION, "summary": generation_summary(records), "last_update": _dt.datetime.utcnow().isoformat()+"Z"})


def run_generation() -> List[Dict[str, Any]]:
    existing = _read_jsonl(MUSIC_OUTPUT_PATH)
    print("existing records:", generation_summary(existing))
    print("benchmark output JSONL:", MUSIC_OUTPUT_PATH.resolve())
    print("candidate queue JSON: disabled in v37; only records/audit/checkpoint are written")

    cache = build_candidate_facet_cache()
    specs = build_candidate_queue(cache)
    specs_by_level = {level: [s for s in specs if s.get("level") == level] for level in MUSIC_TARGET_PER_LEVEL}

    audit = _fresh_audit()
    rejected_sigs: Set[str] = set()
    for e in audit.get("events", [])[-8000:]:
        if e.get("reject_reason") and e.get("reject_reason") not in {"wdqs_error"}:
            c = e.get("constraints")
            if isinstance(c, dict):
                rejected_sigs.add(constraints_signature(c))

    for level in ["L1", "L2", "L3", "L4", "L5"]:
        target = MUSIC_TARGET_PER_LEVEL[level]
        have = level_count(existing, level)
        if have >= target:
            print(f"SKIP music:{level} already has {have}/{target}")
            continue
        need = target - have
        level_specs = specs_by_level.get(level, [])
        print(f"music:{level} target need: {need}; candidate queue: {len(level_specs)}")
        tried = 0
        accepted_this = 0
        skipped_precheck = 0
        pbar = tqdm(total=need, desc=f"music:{level}")
        for spec in level_specs:
            if level_count(existing, level) >= target or target_done(existing):
                break
            if tried >= MAX_ATTEMPTS_PER_LEVEL.get(level, 500):
                print(f"WARNING music:{level}: attempt cap reached ({tried}); accepted_this_run={accepted_this}; have={level_count(existing, level)}/{target}")
                break
            ok, reason = spec_precheck(spec, existing, rejected_sigs)
            if not ok:
                skipped_precheck += 1
                continue
            tried += 1
            record, event = build_record_from_spec(spec, existing)
            event.update({"level": level, "answer_kind": spec.get("answer_kind_quota_bucket"), "constraints": spec.get("constraints"), "ts": _dt.datetime.utcnow().isoformat()+"Z"})
            if record is None:
                audit.setdefault("events", []).append(event)
                rejected_sigs.add(constraints_signature(spec["constraints"]))
            else:
                record["id"] = next_id_for_level(existing, level)
                _append_jsonl(MUSIC_OUTPUT_PATH, record)
                existing.append(record)
                accepted_this += 1
                audit.setdefault("events", []).append({**event, "accepted": True, "id": record["id"]})
                pbar.update(1)
                print(f"OK music:{level} {level_count(existing, level)}/{target}; total={len(existing)}; gold={len(record.get('gold_answer_qids', []))}; tpl={record.get('template_id')}")
            if tried % 5 == 0 or accepted_this % 3 == 0:
                _save_generation_state(audit, existing)
            pbar.set_postfix({"tried": tried, "ok": accepted_this, "pre": skipped_precheck, "total": len(existing)})
        pbar.close()
        print(f"music:{level} finished: have={level_count(existing, level)}/{target}; accepted_this_run={accepted_this}; tried={tried}; precheck_skipped={skipped_precheck}")
        _save_generation_state(audit, existing)

    audit["summary"] = generation_summary(existing)
    audit["finished_at"] = _dt.datetime.utcnow().isoformat()+"Z"
    _json_dump(MUSIC_AUDIT_PATH, audit)
    if not target_done(existing) and not MUSIC_ALLOW_PARTIAL_OUTPUT:
        raise RuntimeError(f"Generation incomplete: {generation_summary(existing)}")
    print("final:", generation_summary(existing))
    return existing

records = run_generation()


existing records: {'total': 0, 'by_level': {}, 'by_kind': {}, 'by_template': {}, 'gold_min': 0, 'gold_max': 0}
benchmark output JSONL: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/music_albums.jsonl
candidate queue JSON: disabled in v37; only records/audit/checkpoint are written
candidate facets built in memory: no WDQS discovery, no candidate JSON; facet keys: 60
candidate specs in memory: 11815 {'L1': 107, 'L2': 1530, 'L3': 894, 'L4': 3466, 'L5': 5818}
candidate templates top: [('music_l5_label_country_genre_performer_other_period', 5625), ('music_l4_country_genre_performer_other_period', 1740), ('music_l4_label_genre_performer_other_period', 1680), ('music_l2_kind_genre_performer_country_period', 573), ('music_l3_release_genre_performer_genre_period', 504), ('music_l2_kind_genre_label_period', 479), ('music_l2_kind_genre_language_period', 478), ('music_l3_label_country_period', 300), ('music_l5_performer_three_release_cond

music:L1:   7%|▋         | 1/15 [00:05<01:17,  5.57s/it, tried=2, ok=1, pre=0, total=1]

OK music:L1 1/15; total=1; gold=55; tpl=music_l1_curated_genre_bridge_period


music:L1:   7%|▋         | 1/15 [01:05<01:17,  5.57s/it, tried=8, ok=1, pre=0, total=1]

In [ ]:
# -------------------------
# Validation / inspection
# -------------------------
def validate_music_records(path: Path = MUSIC_OUTPUT_PATH) -> Dict[str, Any]:
    rows = _read_jsonl(path)
    problems = []
    ids = [r.get("id") for r in rows]
    if len(ids) != len(set(ids)):
        problems.append({"issue": "duplicate_ids", "count": len(ids) - len(set(ids))})
    sigs = [constraints_signature(r.get("constraints", {})) for r in rows]
    dup_sigs = [s for s, n in Counter(sigs).items() if n > 1]
    if dup_sigs:
        problems.append({"issue": "duplicate_constraints", "count": len(dup_sigs)})
    for r in rows:
        meta = r.get("gold_collection_meta") or {}
        if len(r.get("gold_answer_qids", [])) < int(r.get("requested_count", 0)):
            problems.append({"id": r.get("id"), "issue": "gold_less_than_requested"})
        if len(r.get("gold_answer_qids", [])) != len(set(r.get("gold_answer_qids", []))):
            problems.append({"id": r.get("id"), "issue": "duplicate_gold_qids"})
        if not (len(r.get("gold_answer_qids", [])) == len(r.get("gold_answer_labels_ru", [])) == len(r.get("gold_answer_labels_en", []))):
            problems.append({"id": r.get("id"), "issue": "label_length_mismatch"})
        if r.get("constraints") != (r.get("local_validator") or {}).get("filters"):
            problems.append({"id": r.get("id"), "issue": "constraints_local_validator_mismatch"})
        if r.get("constraints") != meta.get("constraints"):
            problems.append({"id": r.get("id"), "issue": "constraints_meta_mismatch"})
        for k, v in (r.get("constraints") or {}).items():
            if re.search(r"\bQ\d+\b|wd:|P\d+", str(v)) or re.search(r"\bQ\d+\b|wd:|P\d+", str(k)):
                problems.append({"id": r.get("id"), "issue": "dirty_constraint", "key": k, "value": v})
        qids = r.get("gold_answer_qids", [])
        if meta.get("complete_count_with_en_label") is not None and int(meta.get("complete_count_with_en_label")) != len(qids):
            problems.append({"id": r.get("id"), "issue": "complete_count_mismatch"})
        if meta.get("diagnostic_count_without_label_filter") is not None and int(meta.get("diagnostic_count_without_label_filter")) != len(qids):
            problems.append({"id": r.get("id"), "issue": "unlabeled_extra_or_count_mismatch"})
        if "wd:Q" not in r.get("sparql_query", "") or "wdt:P31/wdt:P279*" not in r.get("sparql_query", ""):
            problems.append({"id": r.get("id"), "issue": "missing_kind_sparql_pattern"})
    # Report high gold overlap pairs for final curation.
    overlap_pairs = []
    for i in range(len(rows)):
        s = set(rows[i].get("gold_answer_qids") or [])
        for j in range(i+1, len(rows)):
            t = set(rows[j].get("gold_answer_qids") or [])
            if not s or not t:
                continue
            inter = len(s & t); union = len(s | t)
            jac = inter/union if union else 0
            contain = inter/min(len(s), len(t)) if min(len(s), len(t)) else 0
            if jac >= 0.70 or contain >= 0.90:
                overlap_pairs.append({"id1": rows[i].get("id"), "id2": rows[j].get("id"), "jaccard": round(jac, 3), "containment": round(contain, 3), "intersection": inter})
    report = {
        "path": str(path),
        "total": len(rows),
        "by_level": dict(Counter(r.get("complexity") for r in rows)),
        "by_kind": dict(Counter((r.get("gold_collection_meta") or {}).get("answer_kind_quota_bucket") for r in rows)),
        "by_template_top": Counter(r.get("template_id") for r in rows).most_common(30),
        "gold_min": min([len(r.get("gold_answer_qids", [])) for r in rows], default=0),
        "gold_max": max([len(r.get("gold_answer_qids", [])) for r in rows], default=0),
        "problems": problems,
        "high_gold_overlap_pairs": overlap_pairs[:100],
    }
    _json_dump(DOMAIN_OUT_DIR / "music_albums_validation_report.json", report)
    return report

report = validate_music_records()
print(json.dumps(report, ensure_ascii=False, indent=2)[:5000])
